# Janis + Agilent E4980A Control Notebook

Interactive notebook for impedance spectroscopy measurements with temperature and bias control.

**Features:**
- ✅ Clean utility functions for stacked measurements
- ✅ Fixed temperature with bias sweep
- ✅ Temperature sweep (frequency sweep at each temperature)
- ✅ Full temperature + bias sweep (frequency sweep at each temp/bias combination)

**Setup required:**
1. Run `pip install -e .` in `instrument-interfaces/` directory (one time only)
2. Check GPIB addresses match your hardware (see cell 2)
3. Run cells sequentially (Shift+Enter)

In [4]:
# Imports
from nfoinstruments.drivers import Janis, E4890A
from nfoinstruments.drivers.setup import MeasurementSetup
from nfoinstruments.procedures import set_temperature_and_wait, sweep_frequency_lcr, set_bias_and_wait
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from pathlib import Path
# Counter for run numbering
run_count = 1

In [ ]:
# Initialize instruments
mm = MeasurementSetup()

# Connect to devices (update GPIB addresses if needed)
mm.connect_to_devices({
    'GPIB0::16::INSTR': Janis,
    'GPIB0::17::INSTR': E4890A
})

janis = mm.devices['GPIB0::16::INSTR']
lcr = mm.devices['GPIB0::17::INSTR']

print(f"✓ Connected - Janis: {janis.temperature:.1f} K, LCR: Ready")

In [ ]:
# Configure LCR meter
lcr.measurement_type = E4890A.MeasurementType.ZTD  # Impedance magnitude & phase
lcr.measurement_time = E4890A.MeasurementTime.MEDIUM
lcr.signal_amplitude = 0.05  # 50 mV
lcr.averages = 5
lcr.bias = 0.0

print(f"✓ LCR configured: {lcr.measurement_type.name}, {lcr.signal_amplitude}V, {lcr.averages} avg")

In [ ]:
# Define frequency sweep points
frequency_points = np.round(
    np.logspace(np.log10(20), np.log10(2_000_000), num=100), 
    decimals=2
)

print(f"✓ Frequency sweep: {len(frequency_points)} points from {frequency_points[0]} Hz to {frequency_points[-1]} Hz")

---
## Option 1: Single Temperature Measurement

In [ ]:
# Single temperature measurement
target_temp = 300  # K

# Set temperature and wait for stability (with extra settling time)
actual_temp = set_temperature_and_wait(janis, target_temp, extra_settle_time=30, verbose=True)

# Filename format for your data import
filename = f"data/run{run_count:03d}_temp_{actual_temp:.0f}.csv"
print(f"\nMeasuring to: {filename}")

with open(filename, "w") as f:
    f.write("# time,bias,frequency,NA,Z,theta\n")
    sweep_frequency_lcr(janis, lcr, frequency_points, f, verbose=True)

run_count += 1
print(f"\n✓ Complete! Next run: {run_count}")

AttributeError: 'MeasurementSetup' object has no attribute 'janis'

---
## Option 2: Temperature Sweep

In [ ]:
# Temperature sweep (multiple temperatures)
temp_points = [300, 280, 260, 240, 220, 200]  # K

for target_temp in temp_points:
    print(f"\n{'='*60}")
    print(f"Temperature: {target_temp} K")
    print('='*60)
    
    # Set temperature and wait for stability
    actual_temp = set_temperature_and_wait(janis, target_temp, extra_settle_time=30, verbose=True)
    
    # Filename format
    filename = f"data/run{run_count:03d}_temp_{actual_temp:.0f}.csv"
    print(f"\nMeasuring to: {filename}")
    
    with open(filename, "w") as f:
        f.write("# time,bias,frequency,NA,Z,theta\n")
        sweep_frequency_lcr(janis, lcr, frequency_points, f, verbose=True)
    
    run_count += 1
    print(f"✓ Run {run_count-1} complete!")

print(f"\n{'='*60}")
print(f"✓ Temperature sweep complete! Next run: {run_count}")
print('='*60)

---
## Option 3: Fixed Temperature, Bias Sweep

In [ ]:
# Fixed temperature, bias sweep
target_temp = 300  # K
bias_points = [-1.0, -0.5, 0.0, 0.5, 1.0]  # V

print(f"{'='*60}")
print(f"Fixed Temperature Bias Sweep: {target_temp} K")
print('='*60)

# Set temperature and wait for stability
actual_temp = set_temperature_and_wait(janis, target_temp, extra_settle_time=30, verbose=True)

# Loop over bias voltages
for bias in bias_points:
    print(f"\n  Bias: {bias:+.2f} V")
    
    # Set bias and wait for settling
    set_bias_and_wait(lcr, bias, settle_time=1.0)
    
    # Filename format
    filename = f"data/run{run_count:03d}_temp_{actual_temp:.0f}_bias_{bias:+.2f}.csv"
    
    with open(filename, "w") as f:
        f.write("# time,bias,frequency,NA,Z,theta\n")
        sweep_frequency_lcr(janis, lcr, frequency_points, f, verbose=True)
    
    print(f"  ✓ Saved: run{run_count:03d}")
    run_count += 1

print(f"\n{'='*60}")
print(f"✓ Bias sweep complete! Next run: {run_count}")
print('='*60)

---
## Option 4: Temperature Sweep with Bias Sweep at Each Temperature

In [ ]:
# Temperature + Bias sweep (stacked measurement)
temp_points = [300, 280, 260]  # K
bias_points = [-1.0, -0.5, 0.0, 0.5, 1.0]  # V

print(f"{'='*60}")
print(f"Temperature + Bias Sweep")
print(f"Temperatures: {temp_points}")
print(f"Bias points: {bias_points}")
print('='*60)

# Loop over temperatures
for target_temp in temp_points:
    print(f"\n{'='*60}")
    print(f"Temperature: {target_temp} K")
    print('='*60)
    
    # Set temperature and wait for stability
    actual_temp = set_temperature_and_wait(janis, target_temp, extra_settle_time=30, verbose=True)
    
    # Loop over bias voltages at this temperature
    for bias in bias_points:
        print(f"\n  Bias: {bias:+.2f} V")
        
        # Set bias and wait
        set_bias_and_wait(lcr, bias, settle_time=1.0)
        
        # Filename format
        filename = f"data/run{run_count:03d}_temp_{actual_temp:.0f}_bias_{bias:+.2f}.csv"
        
        with open(filename, "w") as f:
            f.write("# time,bias,frequency,NA,Z,theta\n")
            sweep_frequency_lcr(janis, lcr, frequency_points, f, verbose=True)
        
        print(f"  ✓ Saved: run{run_count:03d}")
        run_count += 1

print(f"\n{'='*60}")
print(f"✓ ALL MEASUREMENTS COMPLETE! Next run: {run_count}")
print('='*60)

In [ ]:
# Load a single measurement file
import glob
import os

# Make sure data directory exists
os.makedirs("data", exist_ok=True)

# Find latest file
files = sorted(glob.glob("data/run*.csv"))

if files:
    latest_file = files[-1]
    print(f"Loading: {latest_file}")
    
    # Read CSV, skipping comment lines
    df = pd.read_csv(latest_file, comment='#', names=['time', 'bias', 'frequency', 'NA', 'Z', 'theta'])
    
    print(f"\nData shape: {df.shape}")
    print(f"Bias: {df['bias'].mean():.3f} V")
    print(f"Frequency range: {df['frequency'].min():.0f} - {df['frequency'].max():.0f} Hz")
    print(f"\nFirst few rows:")
    print(df.head())
else:
    print("No data files found yet - run a measurement first!")

In [ ]:
# Plot impedance magnitude and phase vs frequency
if 'df' in locals():
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Impedance magnitude
    ax1.loglog(df['frequency'], df['Z'], 'b-', linewidth=2)
    ax1.set_xlabel('Frequency (Hz)', fontsize=12)
    ax1.set_ylabel('|Z| (Ω)', fontsize=12)
    ax1.set_title(f"Impedance (Bias={df['bias'].mean():.2f}V)", fontsize=13)
    ax1.grid(True, alpha=0.3, which='both')
    
    # Phase
    ax2.semilogx(df['frequency'], df['theta'], 'r-', linewidth=2)
    ax2.set_xlabel('Frequency (Hz)', fontsize=12)
    ax2.set_ylabel('Phase θ (°)', fontsize=12)
    ax2.set_title(f"Phase (Bias={df['bias'].mean():.2f}V)", fontsize=13)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("Load data first (run cell above)")

---
## Quick Reference & Status Check

**Utility Functions Used:**
- `set_temperature_and_wait(controller, temp, extra_settle_time, verbose)` - Set temp and wait for stability
- `set_bias_and_wait(lcr, bias, settle_time)` - Set DC bias and wait
- `sweep_frequency_lcr(controller, lcr, frequencies, file, verbose)` - Perform frequency sweep

**Common Measurement Types:**
- `E4890A.MeasurementType.ZTD` - Impedance magnitude |Z| and phase θ
- `E4890A.MeasurementType.RX` - Resistance R and reactance X
- `E4890A.MeasurementType.CPD` - Parallel capacitance Cp and dissipation D

**Measurement Times:**
- `SHORT` - Fast, less accurate
- `MEDIUM` - Balanced (recommended)
- `LONG` - Slow, most accurate

In [ ]:
# Temperature + Bias sweep
temp_points = [300, 250, 200]  # K
bias_points = np.linspace(0, 1, 6)  # 0 to 1V in 0.2V steps

for target_temp in temp_points:
    print(f"\n{'='*50}")
    print(f"Temperature: {target_temp} K")
    print('='*50)
    
    janis.temperature_setpoint = target_temp
    print("Waiting for stability...")
    
    while not janis.temperature_stable:
        print(f"  Current: {janis.temperature:.1f} K")
        time.sleep(10)
    
    actual_temp = janis.temperature
    print(f"✓ Stable at {actual_temp:.1f} K")
    
    for bias in bias_points:
        print(f"\n  Bias: {bias:.2f} V")
        lcr.bias = bias
        time.sleep(0.5)
        
        # Filename format: run###_temp_###_bias_#.##.csv (matches your import regex)
        filename = f"C:/Users/F110216/Documents/DATA/Horatio/run{run_count}_temp_{actual_temp:.0f}_bias_{bias:.2f}.csv"
        
        with open(filename, "w") as f:
            f.write("# time,bias,frequency,NA,Z,theta\n")
            sweep_frequency_lcr(janis, lcr, frequency_points, f)
        
        print(f"  ✓ Saved: run{run_count}")
        run_count += 1

print(f"\n{'='*50}")
print(f"ALL DONE! Next run: {run_count}")
print('='*50)

In [ ]:
# Check current instrument status
print("=== Janis Status ===")
print(f"Current temperature: {janis.temperature:.2f} K")
print(f"Setpoint: {janis.temperature_setpoint} K")
print(f"Stable: {janis.temperature_stable}")
print(f"Max heater power: {janis.max_heater_power}%")

print("\n=== LCR Meter Status ===")
print(f"Measurement type: {lcr.measurement_type.name}")
print(f"Signal amplitude: {lcr.signal_amplitude} V")
print(f"Averages: {lcr.averages}")
print(f"Bias: {lcr.bias} V")
print(f"Current frequency: {lcr.frequency} Hz")

print(f"\n=== Run Counter ===")
print(f"Next run number: {run_count:03d}")